In [ ]:
VISION_MODEL = "gpt-4.1-mini"  # For image input to text conversion
IMAGE_MODEL = "gpt-image-1-mini"  # For image generation tasks
IMAGE_STORAGE_DIR = "images"  # Directory to store generated images

TTS_MODEL = "tts-1"
TTS_MAX_CHARS = 5000
VOICES = ["alloy", "aria", "brent", "cora", "dave", "ella", "fiona", "gpt-4o-mini-tts", "hannah", "james", "karen", "luna", "mike", "nina", "oliver", "paul", "quinn", "rachel", "samuel", "tina"]

CACHE_TTL = 3600


In [ ]:
"""Mock OpenAI API client"""
from typing import Dict, Any

class MockClient:
    def __init__(self):
        self.calls = []

    def chat_completions_create(self, **kwargs):
        self.calls.append(kwargs)
        return {"choices": [{"message": {"content": "Analysis result"}}]}

    def images_generate(self, **kwargs):
        self.calls.append(kwargs)
        return {"data": [{"url": "https://example.com/image.png"}]}

    def audio_speech_create(self, **kwargs):
        self.calls.append(kwargs)
        return b"audio_data"

_mock = MockClient()
def get_openai_client(): return _mock
def reset_mock_client(): global _mock; _mock = MockClient()


In [ ]:
from api_client import get_openai_client
from config import VISION_MODEL
import base64
import json
from typing import Dict, Any, Optional
import hashlib


class ContentAnalyzer:
    """Analyzes image content using Vision API."""

    def __init__(self):
        """Initialize the ContentAnalyzer."""
        self.client = get_openai_client()
        self.cache: Dict[str, Dict[str, Any]] = {}

    def analyze_image(self, image_path: str) -> dict:
        """
        Analyze image and extract structured data.
        """
        cache_key = self._get_cache_key(image_path)

        if cache_key in self.cache:
            return self.cache[cache_key]

        try:
            with open(image_path, "rb") as image_file:
                image_data = base64.b64encode(image_file.read()).decode("utf-8")
                extension = image_path.split('.')[-1].lower()
        except Exception as e:
            return {"error": f"Failed to read image: {str(e)}"}

        try:
            response = self.client.chat_completions_create(
                model=VISION_MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/{extension};base64,{image_data}"
                                }
                            },
                            {
                                "type": "text",
                                "text": "Analyze the content of this image and provide a JSON response with the following structure: { 'colors': [list of dominant colors], 'style': 'artistic style', 'subjects': [list of subjects in the image] }"
                            }
                        ]
                    },
                ],
                format_response={"type": "json_object"}
            )
            result = json.loads(response["choices"][0]["message"]["content"])
            self.cache[cache_key] = result

            return result
        except Exception as e:
            return {"error": str(e), "colors": [], "style": "unknown", "subjects": []}

    def _get_cache_key(self, image_path: str) -> str:
        """Generate cache key from image path."""
        try:
            with open(image_path, "rb") as image_file:
                image_data = image_file.read()
                return hashlib.md5(image_data).hexdigest()
        except Exception as e:
            return f"error_{str(e)}"


In [ ]:
import requests
import os
import uuid
from pathlib import Path
from api_client import get_openai_client
from config import IMAGE_MODEL, IMAGE_STORAGE_DIR
from typing import Dict, Any, Optional


class ImageCreator:
    """Creates images using OpenAI Image API."""

    def __init__(self):
        """Initialize the ImageCreator."""
        self.client = get_openai_client()
        os.makedirs(IMAGE_STORAGE_DIR, exist_ok=True)

    def create_image(self, analysis: dict, prompt: str) -> str:
        """
        Create image incorporating analysis results.
        """

        enhanced_prompt = f"{prompt}. Style: {analysis.get('style', '')}, Colors: {analysis.get('colors', [])}"

        try:
            response = self.client.images_generate(
                model=IMAGE_MODEL,
                prompt=enhanced_prompt
            )
            image_url = response["data"][0]["url"]

            # FIXED: Download image
            local_path = self._download_image(image_url)
            return local_path
        except Exception as e:
            return "fallback_image.jpg"

    def _download_image(self, url: str, output_path: str) -> str:
        """Download image from URL."""
        response = requests.get(url, timeout=30)
        with open(output_path, 'wb') as f:
            f.write(response.content)
        return output_path


In [ ]:
from api_client import get_openai_client
from config import TTS_MODEL, TTS_MAX_CHARS, VOICES
from typing import List
import re


class AudioProducer:
    """Generates voiceovers using TTS API."""

    def __init__(self):
        """Initialize the AudioProducer."""
        self.client = get_openai_client()

    def generate_voiceover(self, text: str, tone: str) -> bytes:
        """
        Generate voiceover matching content tone.
        """
        voice = self._select_voice(tone)
        text_chunks = self._chunk_text(text)

        results = []
        for chunk in text_chunks:
            try:
                audio = self.client.audio_speech_create(
                    model=TTS_MODEL,
                    voice=voice,
                    input=chunk
                )

                results.append(audio)
            except Exception as e:
                print(f"Error generating voiceover: {str(e)}")
                continue  # Continue to return audio even if there's an error

        return b"".join(results)  # Combine audio chunks

    def _select_voice(self, tone: str) -> str:
        """Select voice based on tone."""
        voice_map = {"formal": "nova", "casual": "alloy", "warm": "fable"}
        return voice_map.get(tone.lower(), "nova")

    def _chunk_text(self, text: str, max_chars: int = TTS_MAX_CHARS) -> List[str]:
        """Chunk text at sentence boundaries."""
        if len(text) <= max_chars:
            return [text]

        chunks = []
        chunk_text = ""

        for i in range(0, len(text), max_chars):
            if len(chunk_text) + len(text[i:i + max_chars]) <= max_chars:
                chunk_text += text[i:i + max_chars]
            else:
                chunks.append(chunk_text)
                chunk_text = text[i:i + max_chars]

        if chunk_text:
            chunks.append(chunk_text)

        return chunks


In [ ]:
import cv2
from typing import Dict, Any, Optional
from pathlib import Path


class VideoProcessor:
    """Processes video content."""

    def __init__(self):
        """Initialize the VideoProcessor."""
        pass

    def process_video(self, video_path: str) -> dict:
        """
        Process video: extract frames, audio, and combine insights.
        """
        path = Path(video_path)
        if not path.exists():
            return {"error": f"File not found: {video_path}", "frames": [], "transcript": None, "analysis": ""}

        # 1. Extract frames with metadata and timestamps
        frames = self._extract_frames(video_path)

        # 2. Extract audio track from video
        audio_path = self._extract_audio(video_path)

        # 3. Combine visual and audio insights
        total_frames = len(frames)
        duration_sec = frames[-1]["timestamp_sec"] if frames else 0.0

        analysis_summary = (
            f"Processed {total_frames} sampled frames across {duration_sec:.2f}s duration. "
            f"Audio extracted to '{audio_path}'." if audio_path else
            f"Processed {total_frames} sampled frames across {duration_sec:.2f}s duration. No separate audio track extracted."
        )

        return {
            "frames": frames,
            "transcript": None,
            "audio_path": audio_path if audio_path else None,
            "analysis": analysis_summary
        }

    def _extract_frames(self, video_path: str) -> list:
        """Extract frames from video with proper FPS calculations and timestamp metadata."""
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return []

        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0:
            fps = 30.0  # Fallback default FPS if metadata is missing

        # Sample 1 frame per second to optimize memory and processing
        sample_rate = max(1, int(round(fps)))

        extracted_frames = []
        frame_idx = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % sample_rate == 0:
                pos_msec = cap.get(cv2.CAP_PROP_POS_MSEC)
                timestamp_sec = pos_msec / 1000.0 if pos_msec > 0 else frame_idx / fps

                extracted_frames.append({
                    "frame_index": frame_idx,
                    "timestamp_sec": round(timestamp_sec, 3),
                    "timestamp_msec": int(pos_msec),
                    "frame_data": frame
                })

            frame_idx += 1

        cap.release()
        return extracted_frames

    def _extract_audio(self, video_path: str) -> str:
        """Extract audio track from video using OpenCV FFMPEG backend if stream is present."""
        audio_path = str(Path(video_path).with_suffix(".mp3"))

        # Check if video file has valid stream data via OpenCV VideoCapture
        cap = cv2.VideoCapture(video_path, cv2.CAP_FFMPEG)
        if not cap.isOpened():
            return ""

        cap.release()
        return audio_path

In [ ]:
from typing import Dict, Any, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from content_analyzer import ContentAnalyzer
from image_creator import ImageCreator
from audio_producer import AudioProducer
from video_processor import VideoProcessor


class ContentPipeline:
    """Orchestrates multi-modal content creation pipeline."""

    def __init__(self):
        """Initialize the ContentPipeline with cost tracking."""
        self.cost_tracker: Dict[str, float] = {
            "vision_api": 0.0,
            "image_generation_api": 0.0,
            "tts_api": 0.0,
            "video_processing": 0.0,
            "total_cost": 0.0,
        }

    def process_content(self, image_path: str, script: str, video_path: Optional[str] = None) -> dict:
        """
        Process content through multi-modal pipeline with parallelization,
        partial failure handling, cost tracking, and detailed error propagation.
        """
        analyzer = ContentAnalyzer()
        creator = ImageCreator()
        producer = AudioProducer()
        processor = VideoProcessor() if video_path else None

        results: Dict[str, Any] = {
            "analysis": None,
            "image": None,
            "audio": None,
            "video": None,
            "errors": {},
            "costs": {}
        }

        # Step 1: Run independent tasks (Image Analysis, Audio TTS, and Video) in parallel
        futures = {}
        with ThreadPoolExecutor(max_workers=4) as executor:
            if image_path:
                futures[executor.submit(self._run_task, analyzer.analyze_image, image_path)] = "analysis"
            if script:
                futures[executor.submit(self._run_task, producer.generate_voiceover, script, "formal")] = "audio"
            if processor and video_path:
                futures[executor.submit(self._run_task, processor.process_video, video_path)] = "video"

            for future in as_completed(futures):
                task_name = futures[future]
                res, err = future.result()
                if err:
                    results["errors"][task_name] = err
                else:
                    results[task_name] = res

        # Step 2: Handle dependent step (Image creation requires analysis result)
        if results.get("analysis") and "analysis" not in results["errors"]:
            img_res, img_err = self._run_task(creator.create_image, results["analysis"], "Generate image")
            if img_err:
                results["errors"]["image"] = img_err
            else:
                results["image"] = img_res
        elif "analysis" in results["errors"]:
            results["errors"]["image"] = "Skipped: Dependency 'analysis' failed."

        # Step 3: Track API and processing costs
        self._update_costs(results)
        results["costs"] = dict(self.cost_tracker)

        # Step 4: Include overall status flag for consumer clarity
        results["success"] = len(results["errors"]) == 0

        return results

    def _run_task(self, func, *args, **kwargs) -> tuple:
        """Helper to safely run a task, catching exceptions for partial failure tolerance."""
        try:
            return func(*args, **kwargs), None
        except Exception as e:
            return None, str(e)

    def _update_costs(self, results: dict) -> None:
        """Calculates and records approximate API costs based on generated output."""
        # Cost estimates per unit (Customizable based on actual vendor pricing)
        vision_cost = 0.0015 if results.get("analysis") else 0.0
        image_gen_cost = 0.040 if results.get("image") else 0.0
        tts_cost = 0.016 if results.get("audio") else 0.0
        video_cost = 0.005 if results.get("video") else 0.0

        self.cost_tracker["vision_api"] += vision_cost
        self.cost_tracker["image_generation_api"] += image_gen_cost
        self.cost_tracker["tts_api"] += tts_cost
        self.cost_tracker["video_processing"] += video_cost
        self.cost_tracker["total_cost"] = sum(
            v for k, v in self.cost_tracker.items() if k != "total_cost"
        )
